In [ ]:
#| default_exp nbskill

## Installing the skill

Install the notebook workflow instructions, then register reusable local repositories.

The repository also ships the instructions that teach an agent how to use these tools. This notebook installs the bundled `SKILL.md` and references into local Codex or Claude skills directories, and can register the nbskill MCP server in Cursor's `mcp.json` so the notebook-first workflow can travel with the package.

The package ships a compact routing skill and optional MCP reference files. The operational guidance lives in `nbskill.skill`, so installation copies package-owned files without rebuilding a skill from a README or changing another checkout.

```python
install_nbskill(skills_dir="~/.codex/skills")
```

### Production contract

Skill installation is Python-first. It installs only into the requested Codex or Claude skills directory, writes MCP configuration only when that target is requested, leaves MCP processes running, returns restart guidance when needed, and installs hooks only when explicitly requested. It does not alter a README or another repository.

In [ ]:
from nbskill.foundation import demo_path, remove_demo_path

In [ ]:
#| export
import json, subprocess, tomllib
from importlib.resources import files
from pathlib import Path

from nbskill.foundation import install_nbdev_pre_commit_hooks

### Installing agent instructions

The installer copies the compact routing skill and its optional MCP reference files. Native notebook methodology lives in `nbskill.skill`, so installation does not maintain another operational playbook or edit a separate checkout.

In [ ]:
#| exporti
def _nbskill_mcp_pids():
    proc = subprocess.run(["ps", "-eo", "pid=,ppid=,command="], text=True, capture_output=True)
    if proc.returncode != 0: return []
    pids = []
    for line in proc.stdout.splitlines():
        parts = line.strip().split(None, 2)
        if len(parts) != 3: continue
        try: pid, ppid = int(parts[0]), int(parts[1])
        except ValueError: continue
        command = parts[2]
        if "nbskill_mcp" in command or "nbskill.mcp" in command: pids.append({"pid": pid, "ppid": ppid, "command": command.strip()})
    return pids

In [ ]:
#| export
def nbskill_mcp_restart_notice(timeout=5.0, force=True, dry_run=False):
    "Return the reconnect instruction; stdio servers are started by the MCP client."
    return {
        "running": bool(_nbskill_mcp_pids()),
        "restarted": False,
        "processes": _nbskill_mcp_pids(),
        "message": "Reconnect the MCP client to start a fresh nbskill MCP server.",
    }

In [ ]:
#| export
def _cursor_mcp_path(workspace=None):
    root = Path(workspace).expanduser() if workspace else Path.home()
    return root / ".cursor" / "mcp.json"

In [ ]:
#| export
def _cursor_nbskill_server_config():
    return {"command": "nbskill_mcp"}

In [ ]:
#| exporti
def _install_cursor_mcp(workspace=None, overwrite=True):
    """Install nbskill's MCP server into Cursor's mcp.json."""
    path = _cursor_mcp_path(workspace)
    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict): raise ValueError(f"Cursor MCP config must be a JSON object: {path}")
    else: data = {}
    servers = data.setdefault("mcpServers", {})
    if not isinstance(servers, dict): raise ValueError(f"Cursor MCP config mcpServers must be an object: {path}")
    expected = _cursor_nbskill_server_config()
    if servers.get("nbskill") == expected:
        return {"installed": False, "reason": "already-current", "path": path}
    if "nbskill" in servers and not overwrite: raise FileExistsError(path)
    servers["nbskill"] = expected
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {
        "installed": True,
        "path": path,
        "server": "nbskill",
        "workspace": str(Path(workspace).expanduser()) if workspace else None,
    }

### Codex MCP configuration

A Codex install writes a project-local configuration in the selected workspace and starts nbskill from that workspace.

In [ ]:
#| export
def _codex_mcp_path(workspace=None):
    root = Path(workspace or ".").expanduser()
    return root / ".codex" / "config.toml"

In [ ]:
#| export
def _codex_nbskill_config(workspace):
    root = Path(workspace).expanduser().resolve()
    lines = [
        "[mcp_servers.nbskill]", "enabled = true", "required = true",
        'command = "nbskill_mcp"', f"cwd = {json.dumps(str(root))}",
        "startup_timeout_sec = 60", "tool_timeout_sec = 180",
    ]
    lines += [
        "", "[mcp_servers.nbskill.tools.edit_notebook]", 'approval_mode = "approve"', "",
        "[mcp_servers.nbskill.tools.context]", 'approval_mode = "approve"',
    ]
    return chr(10).join(lines)

In [ ]:
#| exporti
def _without_codex_nbskill(text):
    """Remove only nbskill MCP tables from Codex TOML text."""
    kept, skipping = [], False
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            table = stripped[1:-1].strip()
            skipping = table == "mcp_servers.nbskill" or table.startswith("mcp_servers.nbskill.")
        if not skipping: kept.append(line)
    return "\n".join(kept).rstrip()

In [ ]:
#| exporti
def _install_codex_mcp(workspace=None, overwrite=True):
    """Install or repair nbskill's MCP server in a project's Codex config."""
    path = _codex_mcp_path(workspace)
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    data = tomllib.loads(text) if text else {}
    servers = data.get("mcp_servers", {})
    if not isinstance(servers, dict): raise ValueError(f"Codex MCP config must be a table: {path}")
    expected_text = _codex_nbskill_config(workspace or ".")
    expected = tomllib.loads(expected_text)["mcp_servers"]["nbskill"]
    if servers.get("nbskill") == expected:
        return {"installed": False, "reason": "already-current", "path": path}
    if "nbskill" in servers and not overwrite: raise FileExistsError(path)
    prefix = _without_codex_nbskill(text)
    updated = (prefix + "\n\n" if prefix else "") + expected_text + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(updated, encoding="utf-8")
    return {"installed": True, "path": path, "server": "nbskill", "workspace": str(Path(workspace or ".").expanduser())}

In [ ]:
#| exporti
def _write_if_changed(path, text):
    old = path.read_text(encoding="utf-8") if path.exists() else ""
    if old == text: return False
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return True

In [ ]:
#| exporti
def _install_skill_tree(root, skill_name, skill_text, references, overwrite=True):
    """Copy changed skill files and leave byte-identical files untouched."""
    dst_dir = root / skill_name
    files_to_install = [(dst_dir / "SKILL.md", skill_text)]
    if references.is_dir():
        files_to_install += [
            (dst_dir / "references" / ref.name, ref.read_text(encoding="utf-8"))
            for ref in references.iterdir() if ref.is_file()
        ]
    changed, unchanged = [], []
    for path,text in files_to_install:
        if path.exists() and path.read_text(encoding="utf-8") != text and not overwrite:
            raise FileExistsError(path)
        if _write_if_changed(path, text): changed.append(path)
        else: unchanged.append(path)
    return changed, unchanged

In [ ]:
#| exporti
def _skill_roots(target, skills_dir):
    target = target.lower()
    if target == "cursor": return target, []
    if skills_dir: return target, [Path(skills_dir).expanduser()]
    if target == "codex": return target, [Path.home() / ".codex" / "skills"]
    if target in {"claude", "claude-code", "claude_code"}: return target, [Path.home() / ".claude" / "skills"]
    if target == "both": return target, [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    raise ValueError("target must be codex, claude, cursor, both, or use skills_dir")

In [ ]:
#| exporti
def _install_skill_files(roots, skill_name, overwrite):
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed, unchanged = [], []
    for root in roots:
        changed, current = _install_skill_tree(root, skill_name, skill_text, references, overwrite)
        installed += changed
        unchanged += current
    return installed, unchanged

In [ ]:
#| export
def install_nbskill(
    target: str = "codex",  # codex, claude, cursor, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Update stale managed files and server entries
    install_hooks: bool = False,  # Install nbdev-clean/nbdev-test pre-commit hooks in the current repo
    restart_mcp: bool = True,  # Report whether the MCP client needs reconnecting
    cursor_workspace: str | None = None,  # Cursor workspace for .cursor/mcp.json; omit for global ~/.cursor/mcp.json
    codex_workspace: str | None = ".",  # Project workspace for .codex/config.toml
    reference_roots: str = "~/projects",  # Local Git repositories to add to the reference index
    index_references: bool = True,  # Index references during installation
):
    """Install current nbskill instructions and optional MCP configuration."""
    target, roots = _skill_roots(target, skills_dir)
    cursor_mcp = {"installed": False, "reason": "target-not-cursor"}
    codex_mcp = {"installed": False, "reason": "target-not-codex"}
    installed, unchanged = _install_skill_files(roots, skill_name, overwrite)
    if target == "cursor": cursor_mcp = _install_cursor_mcp(workspace=cursor_workspace, overwrite=overwrite)
    if not skills_dir and target in {"codex", "both"}:
        codex_mcp = _install_codex_mcp(workspace=codex_workspace, overwrite=overwrite)
    hooks = install_nbdev_pre_commit_hooks(Path.cwd()) if install_hooks else {"installed": False, "reason": "disabled"}
    from nbskill.knowledge import reference_discover
    reference_index = reference_discover(roots=reference_roots, ingest=index_references)
    mcp_notice = {"running": False, "message": "not checked"}
    if restart_mcp and not skills_dir and target in {"codex", "both", "cursor"}:
        mcp_notice = nbskill_mcp_restart_notice()
    for path in installed: print(f"Installed {path}")
    if cursor_mcp.get("installed"): print(f"Installed Cursor MCP config at {cursor_mcp['path']}")
    if codex_mcp.get("installed"): print(f"Installed Codex MCP config at {codex_mcp['path']}")
    if hooks.get("installed"): print(f"Installed nbdev pre-commit hook at {hooks['hook']}")
    if mcp_notice.get("running"):
        print("nbskill MCP restarted:" if mcp_notice.get("restarted") else "nbskill MCP restart incomplete:")
        print(mcp_notice["message"])
    return {
        "installed": installed, "unchanged": unchanged, "cursor_mcp": cursor_mcp,
        "codex_mcp": codex_mcp, "hooks": hooks, "references": reference_index,
        "mcp_restart": mcp_notice,
    }

In [ ]:
#| hide
from unittest.mock import patch

In [ ]:
#| hide
codex_install_root = demo_path("06_skill_codex_wrapper")
try:
    with patch.object(Path, "home", return_value=codex_install_root):
        with patch("nbskill.knowledge.reference_discover", return_value={}):
            result = install_nbskill(
                target="codex", codex_workspace=str(codex_install_root), restart_mcp=False,
                index_references=False,
            )
    server = tomllib.loads((codex_install_root / ".codex" / "config.toml").read_text())["mcp_servers"]["nbskill"]
    assert result["codex_mcp"]["installed"]
    assert "aai_coding" not in result
    assert server["cwd"] == str(codex_install_root.resolve())
    assert "args" not in server
    assert "args" not in tomllib.loads(_codex_nbskill_config(codex_install_root))["mcp_servers"]["nbskill"]
finally: remove_demo_path(codex_install_root)

In [ ]:
#| hide
#| eval: false
install_root = demo_path("06_skill_install")
try:
    with patch("nbskill.knowledge.reference_discover", return_value={}):
        first = install_nbskill(skills_dir=str(install_root), index_references=False)
    skill_dir = install_root / "jupyter-notebooks"
    assert first["installed"]
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "mcp-tools.md").exists()
    assert (skill_dir / "references" / "conversion.md").exists()
    assert (skill_dir / "references" / "extended-tools.md").exists()
    installed_text = {path: path.read_bytes() for path in skill_dir.rglob("*") if path.is_file()}
    with patch("nbskill.knowledge.reference_discover", return_value={}):
        second = install_nbskill(skills_dir=str(install_root), index_references=False)
    assert not second["installed"]
    assert second["unchanged"]
    assert installed_text == {path: path.read_bytes() for path in skill_dir.rglob("*") if path.is_file()}
finally:
    remove_demo_path(install_root)

In [ ]:
#| hide
#| eval: false
cursor_root = demo_path("06_skill_cursor_install")
try:
    (cursor_root / ".cursor").mkdir(parents=True)
    existing = {"mcpServers": {"other": {"command": "python", "args": ["server.py"]}}}
    path = cursor_root / ".cursor" / "mcp.json"
    path.write_text(json.dumps(existing), encoding="utf-8")
    first = _install_cursor_mcp(cursor_root)
    installed_text = path.read_text(encoding="utf-8")
    second = _install_cursor_mcp(cursor_root)
    config = json.loads(installed_text)
    assert first["installed"] and not second["installed"]
    assert second["reason"] == "already-current" and path.read_text(encoding="utf-8") == installed_text
    assert config["mcpServers"]["other"]["command"] == "python"
    assert config["mcpServers"]["nbskill"] == {"command": "nbskill_mcp"}
    assert "args" not in _cursor_nbskill_server_config()
finally:
    remove_demo_path(cursor_root)

In [ ]:
#| hide
#| eval: false
codex_root = demo_path("06_skill_codex_install")
try:
    config_path = codex_root / ".codex" / "config.toml"
    config_path.parent.mkdir(parents=True)
    config_path.write_text(
        "[mcp_servers.other]\ncommand = 'other_mcp'\n\n[mcp_servers.nbskill]\ncommand = 'stale'\n",
        encoding="utf-8",
    )
    first = _install_codex_mcp(codex_root)
    installed_text = config_path.read_text(encoding="utf-8")
    second = _install_codex_mcp(codex_root)
    config = tomllib.loads(installed_text)
    server = config["mcp_servers"]["nbskill"]
    assert first["installed"] and not second["installed"]
    assert second["reason"] == "already-current" and config_path.read_text(encoding="utf-8") == installed_text
    assert config["mcp_servers"]["other"]["command"] == "other_mcp"
    assert server["command"] == "nbskill_mcp" and server["cwd"] == str(codex_root.resolve())
    assert server["tools"]["edit_notebook"]["approval_mode"] == "approve"
    assert server["tools"]["context"]["approval_mode"] == "approve"
finally: remove_demo_path(codex_root)

In [ ]:
#| hide
import tomllib
from pathlib import Path
from fastcore.test import test_eq


In [ ]:
#| hide
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
scripts = tomllib.loads((project_root / "pyproject.toml").read_text(encoding="utf-8"))["project"]["scripts"]
test_eq(scripts["nbskill_mcp"], "nbskill.mcp:main")
test_eq(scripts["install_nbskill"], "nbskill.cli:install_nbskill")
test_eq(scripts["nbskill_mcp_log"], "nbskill.cli:nbskill_mcp_log")
test_eq(scripts["nbskill_mcp_log_problems"], "nbskill.cli:nbskill_mcp_log_problems")


In [ ]:
#| hide
for removed in ("nbskill-mcp", "private-symbol-report", "symbol-graph", "update-cell", "show-doc"):
    for path in [project_root / "README.md", project_root / "nbskill/SKILL.md", *(project_root / "nbskill/references").glob("*.md")]:
        if path.exists(): assert removed not in path.read_text(encoding="utf-8")


In [ ]:
#| hide
#| eval: false
hook_root = demo_path("06_skill_hooks")
try:
    hook_root.mkdir()
    (hook_root / "nbs").mkdir()
    subprocess.run(["git", "init"], cwd=hook_root, check=True, capture_output=True)
    hook_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    pre_commit = hook_root / ".git" / "hooks" / "pre-commit"
    hook_text = pre_commit.read_text(encoding="utf-8")
    assert hook_result["installed"]
    assert "nbdev-clean" in hook_text
    assert "nbdev-test" in hook_text
    assert (hook_root / ".git" / "info" / "nbskill-hooks-installed").exists()

    pre_commit.unlink()
    removed_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    assert not removed_result["installed"]
    assert removed_result["reason"] == "hooks-removed-by-user"
    assert not pre_commit.exists()
    assert (hook_root / ".git" / "info" / "nbskill-hooks-disabled").exists()
finally:
    remove_demo_path(hook_root)


In [ ]:
#| hide
#| eval: false
install_root = demo_path("06_skill_custom_install")
try:
    install_nbskill(
        target="custom", skills_dir=str(install_root), install_hooks=False, restart_mcp=False,
    )
    assert (install_root / "jupyter-notebooks" / "SKILL.md").exists()
    notice = nbskill_mcp_restart_notice(dry_run=True)
    assert "running" in notice and "message" in notice
finally:
    remove_demo_path(install_root)

In [ ]:
#| hide
skill_text = (project_root / "nbskill" / "SKILL.md").read_text(encoding="utf-8")
template_text = (project_root / "nbskill" / "AGENTS.md").read_text(encoding="utf-8")
root_text = (project_root / "AGENTS.md").read_text(encoding="utf-8")
assert "nbskill.skill" in skill_text and "MCP server as the normal interface" not in skill_text
assert "nbskill.skill" in template_text and "mcp__nbskill__" not in template_text
assert "context" in root_text and "reference_query" in root_text
assert "prepare_change" not in root_text and "verify_change" not in root_text